### Agentic RAG With LangGraph

In [ ]:
import os
from typing import TypedDict, List
from langgraph.graph import StateGraph, END
from langchain_groq import ChatGroq
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_classic.schema import Document

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

In [ ]:
# Load GROQ API key from environment or .env
from dotenv import load_dotenv
load_dotenv()
import os
if not os.getenv("GROQ_API_KEY"):
    raise RuntimeError('GROQ_API_KEY not set. Run `export GROQ_API_KEY="gsk-..."` or create a .env file with GROQ_API_KEY=gsk-...')
# Initialize Groq LLM (recommended model: llama-3.1-8b-instant)
print('GROQ_API_KEY present:', bool(os.getenv('GROQ_API_KEY')))
llm = ChatGroq(model="llama-3.1-8b-instant", groq_api_key=os.getenv("GROQ_API_KEY"))

In [ ]:
llm

## State Definition

In [ ]:
class AgentState(TypedDict):
    question: str
    documents: List[Document]
    answer: str
    needs_retrieval: bool

In [ ]:
### Sample Docuemnt And VectorStore
# Sample documents for demonstration
sample_texts = [
    "LangGraph is a library for building stateful, multi-actor applications with LLMs. It extends LangChain with the ability to coordinate multiple chains across multiple steps of computation in a cyclic manner.",
    "RAG (Retrieval-Augmented Generation) is a technique that combines information retrieval with text generation. It retrieves relevant documents and uses them to provide context for generating more accurate responses.",
    "Vector databases store high-dimensional vectors and enable efficient similarity search. They are commonly used in RAG systems to find relevant documents based on semantic similarity.",
    "Agentic systems are AI systems that can take actions, make decisions, and interact with their environment autonomously. They often use planning and reasoning capabilities."
]

documents=[Document(page_content=text) for text in sample_texts]

## Create vector store using local sentence-transformers embeddings (no external API)
try:
    from sentence_transformers import SentenceTransformer
    import numpy as np
    import faiss
except Exception:
    import sys
    print('Installing sentence-transformers and faiss-cpu...')
    sys.executable and __import__('subprocess').check_call([sys.executable, '-m', 'pip', 'install', 'sentence-transformers', 'faiss-cpu'])
    from sentence_transformers import SentenceTransformer
    import numpy as np
    import faiss
# Compute embeddings for documents
model = SentenceTransformer('all-MiniLM-L6-v2')
texts = [d.page_content for d in documents]
vecs = np.array([model.encode(t) for t in texts]).astype('float32')
# Build FAISS index
index = faiss.IndexFlatL2(vecs.shape[1])
index.add(vecs)
# Retriever wrapper
class SimpleFAISSRetriever:
    def __init__(self, index, docs, model):
        self.index = index
        self.docs = docs
        self.model = model
    def invoke(self, query, k=3):
        qv = np.array([self.model.encode(query)]).astype('float32')
        D, I = self.index.search(qv, k)
        return [self.docs[i] for i in I[0]]

retriever = SimpleFAISSRetriever(index, documents, model) #FAISS is the upgraded version of chromaDB

### Agents function

In [ ]:
def decide_retrieval(state: AgentState) -> AgentState:
    """
    Decide if we need to retrieve documents based on the question
    """
    question = state["question"]
    
    # Simple heuristic: if question contains certain keywords, retrieve
    retrieval_keywords = ["what", "how", "explain", "describe", "tell me"]
    needs_retrieval = any(keyword in question.lower() for keyword in retrieval_keywords)
    
    return {**state, "needs_retrieval": needs_retrieval}

In [ ]:
def retrieve_documents(state: AgentState) -> AgentState:
    """
    Retrieve relevant documents based on the question
    """
    question = state["question"]
    documents = retriever.invoke(question)
    
    return {**state, "documents": documents}

In [ ]:
def generate_answer(state: AgentState) -> AgentState:
    """
    Generate an answer using the retrieved documents or direct response
    """
    question = state["question"]
    documents = state.get("documents", [])
    
    if documents:
        # RAG approach: use documents as context
        context = "\n\n".join([doc.page_content for doc in documents])
        prompt = f"""Based on the following context, answer the question:

Context:
{context}

Question: {question}

Answer:"""
    else:
        # Direct response without retrieval
        prompt = f"Answer the following question: {question}"
    
    response = llm.invoke(prompt)
    answer = response.content
    
    return {**state, "answer": answer}

### conditional Logic

In [ ]:
def should_retrieve(state: AgentState) -> str:
    """
    Determine the next step based on retrieval decision
    """
    if state["needs_retrieval"]:
        return "retrieve"
    else:
        return "generate"

### build the Graph

In [ ]:
# Create the state graph
workflow = StateGraph(AgentState)

# Add nodes
workflow.add_node("decide", decide_retrieval)
workflow.add_node("retrieve", retrieve_documents)
workflow.add_node("generate", generate_answer)

# Set entry point
workflow.set_entry_point("decide")

# Add conditional edges
workflow.add_conditional_edges(
    "decide",
    should_retrieve,
    {
        "retrieve": "retrieve",
        "generate": "generate"
    }
)

# Add edges
workflow.add_edge("retrieve", "generate")
workflow.add_edge("generate", END)

# Compile the graph
app = workflow.compile()
app

### test the Agentic System

In [ ]:
def ask_question(question: str):
    """
    Helper function to ask a question and get an answer
    """
    initial_state = {
        "question": question,
        "documents": [],
        "answer": "",
        "needs_retrieval": False
    }
    
    result = app.invoke(initial_state)
    return result

In [ ]:
# Test with a question that should trigger retrieval
question1 = "What is LangGraph?"
result1 = ask_question(question1)
result1

In [ ]:
# Test with another question
question2 = "How does RAG work?"
result2 = ask_question(question2)

print(f"Question: {question2}")
print(f"Retrieved documents: {len(result2['documents'])}")
print(f"Answer: {result2['answer']}")
print("\n" + "="*50 + "\n")